# Orbit Wars — Pure PPO Submission Notebook

**Deliverable.** Trains a MaskablePPO agent, writes a self-contained `main.py`,
packages `submission.tar.gz`, and submits to the `orbit_wars` competition.

Workflow:
1. Install deps
2. (optional) Train locally / on Kaggle GPU → `models/best_model.zip`
3. Upload weights as Kaggle Dataset `orbit-wars-weights`, attach it
4. `%%writefile main.py` (inlined agent — no `env/` imports)
5. Local test vs random (win-rate gate)
6. Package + submit + monitor

> Verified against the real env: Planet `(id,owner,x,y,radius,ships,production)`,
> Fleet `(id,owner,x,y,angle,from_planet_id,ships)`, sun (50,50) r=10,
> `angular_velocity` per-episode in [0.025,0.05], action `[from_id, angle, ships]`,
> config `episodeSteps=500, actTimeout=1s, shipSpeed=6`.


## 1. Setup

In [ ]:
!pip install -q "kaggle-environments>=1.28.0" "stable-baselines3==2.3.2" "sb3-contrib==2.3.2"
import warnings; warnings.filterwarnings('ignore')

## 2. Write the training package (`env/`)

These are the exact, tested modules. Skip this whole section if you only want to
package pre-trained weights for submission.

In [ ]:
import os
os.makedirs("env", exist_ok=True)
open("env/__init__.py","w").close()

In [ ]:
%%writefile env/core.py
"""
env/core.py — physics, observation encoding, action decoding, masking.

Single source of truth shared by the Gymnasium wrapper (training) and the
submission `main.py` (inference). Everything here is verified against the real
`kaggle_environments.envs.orbit_wars` interpreter:

    Planet fields : (id, owner, x, y, radius, ships, production)
    Fleet  fields : (id, owner, x, y, angle, from_planet_id, ships)
    Sun           : center (50, 50), radius 10
    Orbit rule    : a planet rotates iff orbital_radius + radius < 50
    Fleet speed   : 1 + (S-1)*(log(ships)/log(1000))**1.5, capped at shipSpeed(=6)
    angular_velocity: per-episode, uniform in [0.025, 0.05]  (read from obs)
    Action        : list of [from_planet_id, angle_rad, num_ships]
"""
import math
import numpy as np
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

# ── Constants (mirror the interpreter) ────────────────────────────────
SUN_X, SUN_Y, SUN_R = 50.0, 50.0, 10.0
ROTATION_RADIUS_LIMIT = 50.0
MAX_SPEED = 6.0                 # configuration.shipSpeed default
N_PLANETS, N_FLEETS = 40, 20
FRACTIONS = [0.25, 0.50, 0.75, 0.95]

OBS_DIM = 12 * N_PLANETS + 7 * N_FLEETS + 5      # 480 + 140 + 5 = 625
ACTION_NVEC = [10, 40, 4]                         # MultiDiscrete([src, tgt, frac])
MIN_LAUNCH_SHIPS = 5                              # mask src planets below this


# ── Physics ───────────────────────────────────────────────────────────
def fleet_speed(n: int) -> float:
    if n <= 1:
        return 1.0
    s = 1.0 + (MAX_SPEED - 1.0) * (math.log(n) / math.log(1000)) ** 1.5
    return min(s, MAX_SPEED)


def orbital_radius(p) -> float:
    return math.hypot(p.x - SUN_X, p.y - SUN_Y)


def is_orbiting(p) -> bool:
    return (orbital_radius(p) + p.radius) < ROTATION_RADIUS_LIMIT


def predict_pos(p, ang_vel: float, t: int):
    """Position of planet p after t turns. Static planets do not move."""
    if not is_orbiting(p):
        return p.x, p.y
    r = orbital_radius(p)
    a = math.atan2(p.y - SUN_Y, p.x - SUN_X) + ang_vel * t
    return SUN_X + r * math.cos(a), SUN_Y + r * math.sin(a)


def intercept_angle(src, tgt, ships: int, ang_vel: float):
    """Iteratively solve the firing angle to intercept a (possibly) rotating
    target. Returns (angle, predicted_tx, predicted_ty)."""
    tx, ty = tgt.x, tgt.y
    spd = fleet_speed(ships)
    for _ in range(8):
        dist = math.hypot(tx - src.x, ty - src.y)
        t = max(1, math.ceil(dist / spd))
        tx, ty = predict_pos(tgt, ang_vel, t)
    return math.atan2(ty - src.y, tx - src.x), tx, ty


def hits_sun(x1, y1, x2, y2) -> bool:
    """True if segment (x1,y1)->(x2,y2) passes within SUN_R of the sun."""
    dx, dy = x2 - x1, y2 - y1
    den = dx * dx + dy * dy
    if den < 1e-9:
        return math.hypot(x1 - SUN_X, y1 - SUN_Y) < SUN_R
    t = max(0.0, min(1.0, ((SUN_X - x1) * dx + (SUN_Y - y1) * dy) / den))
    return math.hypot(x1 + t * dx - SUN_X, y1 + t * dy - SUN_Y) < SUN_R


# ── Observation encoding ──────────────────────────────────────────────
def _raw_get(raw, key, default):
    """raw may be a dict, kaggle Struct, or SimpleNamespace."""
    if isinstance(raw, dict):
        return raw.get(key, default)
    if hasattr(raw, "get"):
        try:
            return raw.get(key, default)
        except TypeError:
            pass
    return getattr(raw, key, default)


def _encode_planet(p, player, ang_vel, comet_ids, ref_ships) -> list:
    dist = math.hypot(p.x - SUN_X, p.y - SUN_Y)
    t_est = max(1, int(dist / fleet_speed(ref_ships)))
    px, py = predict_pos(p, ang_vel, t_est)
    cur_ang = math.atan2(p.y - SUN_Y, p.x - SUN_X)
    r = orbital_radius(p)
    return [
        (px - SUN_X) / 50.0,                       # predicted rel_x
        (py - SUN_Y) / 50.0,                       # predicted rel_y
        math.sin(cur_ang),                         # current sin_angle
        math.cos(cur_ang),                         # current cos_angle
        min(1.0, r / 50.0),                        # orbit radius (norm)
        min(1.0, p.ships / 500.0),                 # ships (norm)
        p.production / 5.0,                        # production (norm)
        1.0 if p.owner == player else 0.0,         # mine
        1.0 if p.owner not in (-1, player) else 0.0,  # enemy
        1.0 if p.owner == -1 else 0.0,             # neutral
        1.0 if is_orbiting(p) else 0.0,            # orbiting flag
        1.0 if p.id in comet_ids else 0.0,         # comet flag
    ]


def encode_obs(raw_obs) -> np.ndarray:
    planets = [Planet(*p) for p in _raw_get(raw_obs, "planets", [])]
    fleets = [Fleet(*f) for f in _raw_get(raw_obs, "fleets", [])]
    player = _raw_get(raw_obs, "player", 0)
    ang_vel = _raw_get(raw_obs, "angular_velocity", 0.03)
    step = _raw_get(raw_obs, "step", 0)
    comet_ids = set(_raw_get(raw_obs, "comet_planet_ids", []))

    my_pl = [p for p in planets if p.owner == player]
    ref_ships = max((p.ships for p in my_pl), default=50)

    # Planet features — fixed N_PLANETS slots
    pl_feats = []
    for i in range(N_PLANETS):
        if i < len(planets):
            pl_feats.extend(_encode_planet(planets[i], player, ang_vel, comet_ids, ref_ships))
        else:
            pl_feats.extend([0.0] * 12)

    # Fleet features — top N_FLEETS by ship count
    fl_feats = []
    top_fleets = sorted(fleets, key=lambda x: -x.ships)[:N_FLEETS]
    for i in range(N_FLEETS):
        if i < len(top_fleets):
            f = top_fleets[i]
            fl_feats.extend([
                (f.x - SUN_X) / 50.0,
                (f.y - SUN_Y) / 50.0,
                math.sin(f.angle), math.cos(f.angle),
                min(1.0, f.ships / 500.0),
                1.0 if f.owner == player else 0.0,
                1.0 if f.owner != player else 0.0,
            ])
        else:
            fl_feats.extend([0.0] * 7)

    # Global features
    total_pl = max(len(planets), 1)
    my_ships = sum(p.ships for p in planets if p.owner == player)
    all_ships = sum(p.ships for p in planets) + 1e-6
    my_prod = sum(p.production for p in planets if p.owner == player)
    all_prod = sum(p.production for p in planets) + 1e-6
    global_feats = [
        step / 500.0,
        sum(1 for p in planets if p.owner == player) / total_pl,
        my_ships / all_ships,
        my_prod / all_prod,
        _raw_get(raw_obs, "remainingOverageTime", 60.0) / 60.0,
    ]

    vec = np.array(pl_feats + fl_feats + global_feats, dtype=np.float32)
    # Defensive clip — Box space is [-2, 2]
    return np.clip(vec, -2.0, 2.0)


# ── Action masking ────────────────────────────────────────────────────
def get_action_masks(raw_planets, player: int) -> np.ndarray:
    """Bool mask of shape (10 + 40 + 4,). True == valid choice.

    Guarantees at least one valid src so MaskablePPO never sees an empty
    discrete dimension (which would raise)."""
    planets = [Planet(*p) for p in raw_planets]
    my_pl = [p for p in planets if p.owner == player]

    src_mask = np.zeros(10, dtype=bool)
    for i, p in enumerate(my_pl[:10]):
        src_mask[i] = p.ships >= MIN_LAUNCH_SHIPS

    if not src_mask.any():
        # Fallback: pick the planet with the most ships (or slot 0 if none).
        if my_pl:
            best = max(range(min(len(my_pl), 10)), key=lambda i: my_pl[i].ships)
            src_mask[best] = True
        else:
            src_mask[0] = True

    tgt_mask = np.ones(40, dtype=bool)
    frac_mask = np.ones(4, dtype=bool)
    return np.concatenate([src_mask, tgt_mask, frac_mask])


# ── Action decoding ───────────────────────────────────────────────────
def decode_action(action, raw_planets, player: int, ang_vel: float) -> list:
    """Map MultiDiscrete indices -> kaggle action [[from_id, angle, ships]]."""
    planets = [Planet(*p) for p in raw_planets]
    my_pl = [p for p in planets if p.owner == player]
    if not my_pl or not planets:
        return []

    src_idx, tgt_idx, frac_idx = int(action[0]), int(action[1]), int(action[2])
    src = my_pl[src_idx % len(my_pl)]
    tgt = planets[tgt_idx % len(planets)]
    if tgt.id == src.id:
        return []

    ships = max(1, int(src.ships * FRACTIONS[frac_idx % 4]))
    ships = min(ships, src.ships)
    if ships <= 0:
        return []

    angle, tx, ty = intercept_angle(src, tgt, ships, ang_vel)

    # If the straight intercept line clips the sun, nudge the angle to arc past it.
    if hits_sun(src.x, src.y, tx, ty):
        for delta in (math.pi / 12, -math.pi / 12, math.pi / 6, -math.pi / 6):
            a2 = angle + delta
            ex = src.x + 80 * math.cos(a2)
            ey = src.y + 80 * math.sin(a2)
            if not hits_sun(src.x, src.y, ex, ey):
                angle = a2
                break

    return [[src.id, float(angle), int(ships)]]

In [ ]:
%%writefile env/reward.py
"""
env/reward.py — shaped reward, with a stage switch for ablation.

Component map (from architecture.md §3):
    A production_tick  — reward owning productive planets each turn
    B planet_ratio     — fraction of planets held
    C capture_bonus    — capturing a planet (scaled by its production)
    D lose_penalty     — losing a planet (scaled by its production)
    E ship_ratio       — fraction of all ships you own (aligns with win cond.)
    F terminal         — amplified +/- at game end

Stage 1 submission spec (task.md) enables ONLY A + F.
Stage 2+ enables the full A..F set.

The interpreter's terminal `kaggle_reward` is +1 (win / top score) or -1.
"""
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

WEIGHTS = {
    "production_tick": 0.002,   # A
    "planet_ratio":    0.010,   # B
    "capture_bonus":   0.100,   # C
    "lose_penalty":    0.150,   # D
    "ship_ratio":      0.020,   # E
    "terminal":        10.0,    # F
}

# Which components are active. "full" = A..F, "minimal" = A + F only.
_PROFILES = {
    "minimal": {"production_tick", "terminal"},
    "full": set(WEIGHTS.keys()),
}


def _planets(raw):
    if raw is None:
        return []
    pls = raw.get("planets", []) if hasattr(raw, "get") else getattr(raw, "planets", [])
    return [Planet(*p) for p in pls]


def compute_reward(prev_raw, curr_raw, player, done, kaggle_reward, profile="full") -> float:
    active = _PROFILES.get(profile, _PROFILES["full"])

    if curr_raw is None:
        # Episode ended without a follow-up observation.
        if done and "terminal" in active:
            return float(kaggle_reward) * WEIGHTS["terminal"]
        return 0.0

    prev_pl = _planets(prev_raw)
    curr_pl = _planets(curr_raw)

    reward = 0.0

    # A — production tick
    if "production_tick" in active:
        reward += sum(p.production for p in curr_pl if p.owner == player) * WEIGHTS["production_tick"]

    # B — planet ratio
    if "planet_ratio" in active:
        reward += (sum(1 for p in curr_pl if p.owner == player) / max(len(curr_pl), 1)) * WEIGHTS["planet_ratio"]

    # C & D — capture / loss (scaled by production value)
    if "capture_bonus" in active or "lose_penalty" in active:
        prev_mine = {p.id: p for p in prev_pl if p.owner == player}
        curr_mine = {p.id: p for p in curr_pl if p.owner == player}
        if "capture_bonus" in active:
            for pid in set(curr_mine) - set(prev_mine):
                reward += curr_mine[pid].production * WEIGHTS["capture_bonus"]
        if "lose_penalty" in active:
            for pid in set(prev_mine) - set(curr_mine):
                reward -= prev_mine[pid].production * WEIGHTS["lose_penalty"]

    # E — ship ratio
    if "ship_ratio" in active:
        my_ships = sum(p.ships for p in curr_pl if p.owner == player)
        all_ships = sum(p.ships for p in curr_pl) + 1e-6
        reward += (my_ships / all_ships) * WEIGHTS["ship_ratio"]

    # F — terminal
    if done and "terminal" in active:
        reward += float(kaggle_reward) * WEIGHTS["terminal"]

    return float(reward)

In [ ]:
%%writefile env/tb_compat.py
"""
env/tb_compat.py — make importing stable-baselines3 robust.

SB3 unconditionally does `from torch.utils.tensorboard import SummaryWriter`.
On some local images that drags in a broken tensorflow/protobuf pair and the
import raises a non-ImportError (e.g. protobuf VersionError), which SB3 does not
catch. This module probes that import once; if it fails, it registers a
lightweight stub so SB3 imports cleanly. On a healthy image (Kaggle) it is a
no-op.

Import this BEFORE any sb3_contrib / stable_baselines3 import. Use
`TENSORBOARD_OK` to decide whether to pass a real `tensorboard_log` path.
"""
import sys
import types

TENSORBOARD_OK = True

try:  # probe the real writer
    import torch.utils.tensorboard  # noqa: F401
except Exception:  # broken tensorboard/tensorflow/protobuf on this image
    TENSORBOARD_OK = False
    _stub = types.ModuleType("torch.utils.tensorboard")

    class SummaryWriter:  # noqa: N801 - mimic the real name
        def __init__(self, *a, **k):
            raise RuntimeError(
                "tensorboard is unavailable on this image (tb_compat stub); "
                "pass tensorboard_log=None"
            )

    _stub.SummaryWriter = SummaryWriter
    sys.modules["torch.utils.tensorboard"] = _stub

In [ ]:
%%writefile env/orbit_env.py
"""
env/orbit_env.py — Gymnasium wrapper around kaggle_environments `orbit_wars`,
plus helpers to turn a trained model into a kaggle opponent agent.

MaskablePPO needs an `action_masks()` method on the env; we expose it and wrap
with sb3-contrib's ActionMasker.
"""
import os
from env import tb_compat  # noqa: F401  (must precede any sb3 import)
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from kaggle_environments import make
from sb3_contrib.common.wrappers import ActionMasker

from env.core import (
    encode_obs, decode_action, get_action_masks,
    OBS_DIM, ACTION_NVEC,
)
from env.reward import compute_reward

# Builtin string agents understood directly by kaggle_environments.
_BUILTIN = {"random", "starter"}


# ── Model → kaggle opponent agent ─────────────────────────────────────
def make_model_agent(model_path, device="cpu", deterministic=True):
    """Return a kaggle agent callable backed by a saved MaskablePPO model.

    The model is loaded lazily on first call (so the callable stays cheap to
    pickle for SubprocVecEnv / cloudpickle)."""
    from sb3_contrib import MaskablePPO

    state = {"model": None}

    def _agent(obs):
        if state["model"] is None:
            state["model"] = MaskablePPO.load(model_path, device=device)
        planets = obs.get("planets", []) if hasattr(obs, "get") else obs.planets
        player = obs.get("player", 0) if hasattr(obs, "get") else obs.player
        ang_vel = obs.get("angular_velocity", 0.03) if hasattr(obs, "get") else obs.angular_velocity
        vec = encode_obs(obs)
        masks = get_action_masks(planets, player)
        action, _ = state["model"].predict(vec, action_masks=masks, deterministic=deterministic)
        return decode_action(action, planets, player, ang_vel)

    return _agent


def resolve_opponent(opponent):
    """Normalise the `opponent` argument into something env.train accepts.

    Accepts: builtin string ("random"/"starter"), a path to a .zip model, or an
    already-callable agent."""
    if callable(opponent):
        return opponent
    if isinstance(opponent, str):
        if opponent in _BUILTIN:
            return opponent
        if opponent.endswith(".zip") and os.path.exists(opponent):
            return make_model_agent(opponent)
    # Fall back to the string as-is (lets kaggle resolve .py files etc.)
    return opponent


# ── Gym env ───────────────────────────────────────────────────────────
class OrbitWarsEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, opponent="random", reward_profile="full"):
        super().__init__()
        self.opponent = opponent
        self.reward_profile = reward_profile

        self.observation_space = spaces.Box(-2.0, 2.0, (OBS_DIM,), np.float32)
        self.action_space = spaces.MultiDiscrete(ACTION_NVEC)

        self._env = None
        self._trainer = None
        self._prev = None
        self._step = 0

    # ── Gym API ──
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._env = make("orbit_wars", debug=False)
        opp = resolve_opponent(self.opponent)
        self._trainer = self._env.train([None, opp])
        raw = self._trainer.reset()
        self._prev = raw
        self._step = 0
        return encode_obs(raw), {}

    def _prev_field(self, key, default):
        p = self._prev
        if p is None:
            return default
        return p.get(key, default) if hasattr(p, "get") else getattr(p, key, default)

    def step(self, action):
        raw_planets = self._prev_field("planets", [])
        ang_vel = self._prev_field("angular_velocity", 0.03)
        player = self._prev_field("player", 0)

        kaggle_act = decode_action(action, raw_planets, player, ang_vel)
        raw, krew, done, info = self._trainer.step(kaggle_act)
        self._step += 1

        obs = encode_obs(raw) if raw is not None else np.zeros(OBS_DIM, np.float32)
        rew = compute_reward(self._prev, raw, player, done, krew, profile=self.reward_profile)
        self._prev = raw if raw is not None else self._prev
        return obs, rew, bool(done), False, info

    # ── Action masking ──
    def action_masks(self) -> np.ndarray:
        if self._prev is None:
            return np.ones(sum(ACTION_NVEC), dtype=bool)
        raw_planets = self._prev_field("planets", [])
        player = self._prev_field("player", 0)
        return get_action_masks(raw_planets, player)


def make_masked_env(opponent="random", reward_profile="full"):
    """Factory returning an ActionMasker-wrapped OrbitWarsEnv."""
    def _init():
        env = OrbitWarsEnv(opponent=opponent, reward_profile=reward_profile)
        return ActionMasker(env, lambda e: e.action_masks())
    return _init

In [ ]:
os.makedirs("train", exist_ok=True)
open("train/__init__.py","w").close()

In [ ]:
%%writefile train/selfplay.py
"""
train/selfplay.py — ELO-ish self-play opponent pool.

Keeps the most recent snapshots and samples an opponent for each round:
60% newest, 40% uniform across the pool. Paths point at saved .zip models
(consumed by env.orbit_env.make_model_agent via resolve_opponent).
"""
import os
import random


class SelfPlayPool:
    MAX_SIZE = 6

    def __init__(self, seed=None, dir="snapshots/"):
        os.makedirs(dir, exist_ok=True)
        self.dir = dir
        self.paths = [seed] if seed else []

    def save(self, model, round_i):
        path = os.path.join(self.dir, f"round_{round_i:03d}.zip")
        model.save(path)
        self.paths.append(path)
        if len(self.paths) > self.MAX_SIZE:
            self.paths.pop(0)
        return path

    def sample(self) -> str:
        if not self.paths:
            return "random"
        if random.random() < 0.6:
            return self.paths[-1]            # newest
        return random.choice(self.paths)     # uniform from pool

In [ ]:
%%writefile train/curriculum.py
"""
train/curriculum.py — 3-stage curriculum training with MaskablePPO.

    Stage 1: vs random           (reward profile = minimal: A + F)
    Stage 2: vs stage-1 model    (reward profile = full)
    Stage 3: self-play ELO pool  (reward profile = full)

All stage fns accept overrides for total_timesteps / n_envs so the same code
path is used for smoke tests and full runs.
"""
import os
from env import tb_compat  # noqa: F401  (must precede any sb3 import)
from sb3_contrib import MaskablePPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

from env.orbit_env import make_masked_env
from train.selfplay import SelfPlayPool

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

PPO_BASE = dict(
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=512,
    n_epochs=10,
    gamma=0.995,            # 500-turn games -> high discount
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=[256, 256, 128]),
    verbose=1,
    # Disable TB automatically when the image can't provide a working writer.
    tensorboard_log="./tb_logs/" if tb_compat.TENSORBOARD_OK else None,
)

N_ENVS = 8


def build_vec(opponent, reward_profile, n_envs, use_subproc=True):
    """Vectorised env. Falls back to DummyVecEnv for n_envs == 1 or when
    subprocesses are undesirable (debugging / Windows quirks)."""
    factories = [make_masked_env(opponent, reward_profile) for _ in range(n_envs)]
    if use_subproc and n_envs > 1:
        return SubprocVecEnv(factories)
    return DummyVecEnv(factories)


def train_stage1(total_timesteps=5_000_000, n_envs=N_ENVS, use_subproc=True,
                 save_path=None, ppo_overrides=None):
    """vs random — bootstrap from scratch. Reward = A + F (minimal)."""
    vec = build_vec("random", "minimal", n_envs, use_subproc)
    cfg = dict(PPO_BASE)
    if ppo_overrides:
        cfg.update(ppo_overrides)
    model = MaskablePPO("MlpPolicy", vec, **cfg)
    model.learn(total_timesteps, progress_bar=False)
    path = save_path or os.path.join(MODELS_DIR, "stage1_random")
    model.save(path)
    vec.close()
    return model, path


def train_stage2(stage1_path=None, total_timesteps=15_000_000, n_envs=N_ENVS,
                 use_subproc=True, save_path=None):
    """vs stage-1 model — learn orbital intercept. Reward = full."""
    stage1_path = stage1_path or os.path.join(MODELS_DIR, "stage1_random.zip")
    vec = build_vec(stage1_path, "full", n_envs, use_subproc)
    model = MaskablePPO.load(stage1_path.replace(".zip", ""), env=vec)
    model.learning_rate = 1e-4      # fine-tune
    model.ent_coef = 0.01
    model.learn(total_timesteps, reset_num_timesteps=False, progress_bar=False)
    path = save_path or os.path.join(MODELS_DIR, "stage2_rulebased")
    model.save(path)
    vec.close()
    return model, path


def train_stage3(stage2_path=None, rounds=3, steps_per_round=3_000_000,
                 n_envs=N_ENVS, use_subproc=True, save_path=None):
    """Self-play with an ELO pool. Reward = full."""
    stage2_path = stage2_path or os.path.join(MODELS_DIR, "stage2_rulebased.zip")
    pool = SelfPlayPool(seed=stage2_path)

    model = MaskablePPO.load(stage2_path.replace(".zip", ""))
    model.learning_rate = 5e-5
    model.ent_coef = 0.005

    last_round_path = None
    for round_i in range(rounds):
        opp = pool.sample()
        vec = build_vec(opp, "full", n_envs, use_subproc)
        model.set_env(vec)
        model.learn(steps_per_round, reset_num_timesteps=False, progress_bar=False)
        last_round_path = pool.save(model, round_i)
        vec.close()
        print(f"[stage3] round {round_i}: opp={opp} pool={pool.paths}")

    path = save_path or os.path.join(MODELS_DIR, "stage3_selfplay")
    model.save(path)
    # Also save a round-tagged alias matching task.md naming (e.g. _r3).
    model.save(os.path.join(MODELS_DIR, f"stage3_selfplay_r{rounds}"))
    return model, path, last_round_path

## 3. Train

`NB_STEPS` is small by default so the notebook finishes quickly. For real
submissions follow `task.md`: Stage 1 = 5M, Stage 2 = 15M, Stage 3 = self-play.
Train heavy runs locally (CPU MLP) and just upload the weights — the MLP policy
barely benefits from the T4. Set `NB_STEPS` higher only if training in-notebook.

> `tensorboard_log=None` here avoids a tensorboard import on some images; pass
> a path if you want TB curves.

In [ ]:
from train.curriculum import train_stage1

NB_STEPS = 200_000   # bump to 5_000_000 for a real Stage-1 run
model, path = train_stage1(
    total_timesteps=NB_STEPS,
    n_envs=4,
    use_subproc=False,            # notebooks: DummyVecEnv is most robust
    save_path="models/best_model",
    ppo_overrides={"tensorboard_log": None},
)
print("saved:", path + ".zip")

## 4. Upload weights as a Kaggle Dataset (run once)

Create / version a dataset named **`orbit-wars-weights`** containing
`best_model.zip`, then **attach it** to this notebook. The agent loads from
`/kaggle/input/orbit-wars-weights/best_model.zip`.

```bash
# locally, after training:
mkdir -p ow_weights && cp models/best_model.zip ow_weights/
kaggle datasets init -p ow_weights
# edit ow_weights/dataset-metadata.json -> title/id "orbit-wars-weights"
kaggle datasets create -p ow_weights      # first time
kaggle datasets version -p ow_weights -m "new weights"   # updates
```

## 5. Verify the attached weights load

In [ ]:
import os
from sb3_contrib import MaskablePPO

CANDIDATES = [
    "/kaggle/input/orbit-wars-weights/best_model.zip",
    "models/best_model.zip",   # fallback if trained in this session
]
WEIGHTS = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert WEIGHTS, f"No weights found in {CANDIDATES}"
_m = MaskablePPO.load(WEIGHTS, device="cpu")
print("Loaded weights from:", WEIGHTS)
print(_m.policy)

## 6. Write `main.py` (self-contained agent)

Set `OW_MODEL_PATH` so the agent finds the weights both on Kaggle and during the
local test below. The file itself tries the standard `/kaggle/input/...` paths
first, then `OW_MODEL_PATH`.

In [ ]:
import os
os.environ["OW_MODEL_PATH"] = os.path.abspath(WEIGHTS)

In [ ]:
%%writefile main.py
"""
Orbit Wars — Pure PPO Agent (self-contained submission file).

Everything (physics, observation encoding, action masking/decoding, model
loading, agent entry point) is inlined here. Kaggle only sees this file inside
submission.tar.gz — no imports from env/.

Model weights are loaded from an attached Kaggle Dataset. Set OW_MODEL_PATH to
override the path (used for local testing).
"""
import os
import math
import numpy as np
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet
from sb3_contrib import MaskablePPO

# ── Constants ─────────────────────────────────────────────────────────
SUN_X, SUN_Y, SUN_R = 50.0, 50.0, 10.0
ROTATION_RADIUS_LIMIT = 50.0
MAX_SPEED = 6.0
N_PLANETS, N_FLEETS = 40, 20
FRACTIONS = [0.25, 0.50, 0.75, 0.95]
MIN_LAUNCH_SHIPS = 5

# ── Physics ───────────────────────────────────────────────────────────
def fleet_speed(n):
    if n <= 1:
        return 1.0
    s = 1.0 + (MAX_SPEED - 1.0) * (math.log(n) / math.log(1000)) ** 1.5
    return min(s, MAX_SPEED)

def orbital_radius(p):
    return math.hypot(p.x - SUN_X, p.y - SUN_Y)

def is_orbiting(p):
    return (orbital_radius(p) + p.radius) < ROTATION_RADIUS_LIMIT

def predict_pos(p, av, t):
    if not is_orbiting(p):
        return p.x, p.y
    r = orbital_radius(p)
    a = math.atan2(p.y - SUN_Y, p.x - SUN_X) + av * t
    return SUN_X + r * math.cos(a), SUN_Y + r * math.sin(a)

def intercept_angle(src, tgt, ships, av):
    tx, ty = tgt.x, tgt.y
    spd = fleet_speed(ships)
    for _ in range(8):
        d = math.hypot(tx - src.x, ty - src.y)
        t = max(1, math.ceil(d / spd))
        tx, ty = predict_pos(tgt, av, t)
    return math.atan2(ty - src.y, tx - src.x), tx, ty

def hits_sun(x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    den = dx * dx + dy * dy
    if den < 1e-9:
        return math.hypot(x1 - SUN_X, y1 - SUN_Y) < SUN_R
    t = max(0.0, min(1.0, ((SUN_X - x1) * dx + (SUN_Y - y1) * dy) / den))
    return math.hypot(x1 + t * dx - SUN_X, y1 + t * dy - SUN_Y) < SUN_R

# ── Obs helpers ───────────────────────────────────────────────────────
def _get(raw, key, default):
    if isinstance(raw, dict):
        return raw.get(key, default)
    if hasattr(raw, "get"):
        try:
            return raw.get(key, default)
        except TypeError:
            pass
    return getattr(raw, key, default)

# ── Observation ───────────────────────────────────────────────────────
def encode_obs(raw):
    planets = [Planet(*p) for p in _get(raw, "planets", [])]
    fleets = [Fleet(*f) for f in _get(raw, "fleets", [])]
    player = _get(raw, "player", 0)
    av = _get(raw, "angular_velocity", 0.03)
    step = _get(raw, "step", 0)
    cids = set(_get(raw, "comet_planet_ids", []))
    my_pl = [p for p in planets if p.owner == player]
    ref = max((p.ships for p in my_pl), default=50)

    def enc_p(p):
        d = math.hypot(p.x - SUN_X, p.y - SUN_Y)
        t = max(1, int(d / fleet_speed(ref)))
        px, py = predict_pos(p, av, t)
        ca = math.atan2(p.y - SUN_Y, p.x - SUN_X)
        r = orbital_radius(p)
        return [
            (px - SUN_X) / 50.0, (py - SUN_Y) / 50.0,
            math.sin(ca), math.cos(ca),
            min(1.0, r / 50.0), min(1.0, p.ships / 500.0), p.production / 5.0,
            float(p.owner == player), float(p.owner not in (-1, player)),
            float(p.owner == -1), float(is_orbiting(p)), float(p.id in cids),
        ]

    pf = []
    for i in range(N_PLANETS):
        pf.extend(enc_p(planets[i]) if i < len(planets) else [0.0] * 12)

    ff = []
    top = sorted(fleets, key=lambda x: -x.ships)[:N_FLEETS]
    for i in range(N_FLEETS):
        if i < len(top):
            f = top[i]
            ff.extend([(f.x - SUN_X) / 50.0, (f.y - SUN_Y) / 50.0,
                       math.sin(f.angle), math.cos(f.angle),
                       min(1.0, f.ships / 500.0),
                       float(f.owner == player), float(f.owner != player)])
        else:
            ff.extend([0.0] * 7)

    tp = max(len(planets), 1)
    gf = [
        step / 500.0,
        sum(1 for p in planets if p.owner == player) / tp,
        sum(p.ships for p in planets if p.owner == player) / (sum(p.ships for p in planets) + 1e-6),
        sum(p.production for p in planets if p.owner == player) / (sum(p.production for p in planets) + 1e-6),
        _get(raw, "remainingOverageTime", 60.0) / 60.0,
    ]
    return np.clip(np.array(pf + ff + gf, dtype=np.float32), -2.0, 2.0)

# ── Action Masking ────────────────────────────────────────────────────
def get_masks(raw_planets, player):
    planets = [Planet(*p) for p in raw_planets]
    my_pl = [p for p in planets if p.owner == player]
    src_m = np.zeros(10, dtype=bool)
    for i, p in enumerate(my_pl[:10]):
        src_m[i] = p.ships >= MIN_LAUNCH_SHIPS
    if not src_m.any():
        if my_pl:
            best = max(range(min(len(my_pl), 10)), key=lambda i: my_pl[i].ships)
            src_m[best] = True
        else:
            src_m[0] = True
    return np.concatenate([src_m, np.ones(40, bool), np.ones(4, bool)])

# ── Action Decode ─────────────────────────────────────────────────────
def decode_action(action, raw_planets, player, av):
    planets = [Planet(*p) for p in raw_planets]
    my_pl = [p for p in planets if p.owner == player]
    if not my_pl or not planets:
        return []
    src = my_pl[int(action[0]) % len(my_pl)]
    tgt = planets[int(action[1]) % len(planets)]
    if tgt.id == src.id:
        return []
    ships = max(1, min(int(src.ships * FRACTIONS[int(action[2]) % 4]), src.ships))
    if ships <= 0:
        return []
    angle, tx, ty = intercept_angle(src, tgt, ships, av)
    if hits_sun(src.x, src.y, tx, ty):
        for d in (math.pi / 12, -math.pi / 12, math.pi / 6, -math.pi / 6):
            a2 = angle + d
            ex, ey = src.x + 80 * math.cos(a2), src.y + 80 * math.sin(a2)
            if not hits_sun(src.x, src.y, ex, ey):
                angle = a2
                break
    return [[src.id, float(angle), int(ships)]]

# ── Model Loading ─────────────────────────────────────────────────────
_MODEL = None
_CANDIDATE_PATHS = [
    os.environ.get("OW_MODEL_PATH", ""),
    "/kaggle/input/orbit-wars-weights/best_model.zip",
    "/kaggle/input/orbit-wars-weights/stage3_selfplay.zip",
    "/kaggle/input/orbit-wars-weights/stage1_random.zip",
]

def _get_model():
    global _MODEL
    if _MODEL is None:
        for path in _CANDIDATE_PATHS:
            if path and os.path.exists(path):
                _MODEL = MaskablePPO.load(path, device="cpu")
                return _MODEL
        raise FileNotFoundError(
            "No model weights found. Tried: "
            + ", ".join(p for p in _CANDIDATE_PATHS if p)
        )
    return _MODEL

# ── Agent Entry Point ─────────────────────────────────────────────────
def agent(obs, config=None):
    planets = _get(obs, "planets", [])
    player = _get(obs, "player", 0)
    av = _get(obs, "angular_velocity", 0.03)
    vec = encode_obs(obs)
    masks = get_masks(planets, player)
    action, _ = _get_model().predict(vec, action_masks=masks, deterministic=True)
    return decode_action(action, planets, player, av)

## 7. Local test vs random (win-rate gate)

In [ ]:
from kaggle_environments import make

results = {"win": 0, "lose": 0, "draw": 0}
exceptions = 0
N_GAMES = 20
for seed in range(N_GAMES):
    env = make("orbit_wars", configuration={"seed": seed})
    result = env.run(["main.py", "random"])
    last = result[-1]
    for a in last:
        if a.status not in ("ACTIVE", "DONE", "INACTIVE"):
            exceptions += 1
    r0, r1 = last[0].reward, last[1].reward
    if r0 > r1: results["win"] += 1
    elif r0 < r1: results["lose"] += 1
    else: results["draw"] += 1

win_rate = results["win"] / N_GAMES
print(f"Win rate vs random: {win_rate:.1%}  {results}  exceptions={exceptions}")
assert exceptions == 0, "agent raised during episodes"
# For a real Stage-1 model expect >= 0.70 (raise per stage; S2+ -> 0.80).
# assert win_rate >= 0.70, f"below submit threshold: {win_rate:.1%}"

## 8. Package `submission.tar.gz` (must contain only `main.py` at root)

In [ ]:
import tarfile
with tarfile.open("submission.tar.gz", "w:gz") as tar:
    tar.add("main.py", arcname="main.py")
print("contents:")
with tarfile.open("submission.tar.gz") as tar:
    for m in tar.getmembers():
        print(" ", m.name)

## 9. Submit

In [ ]:
SUBMISSION_MESSAGE = "S1 PPO stage1 vs random"
!kaggle competitions submit orbit-wars -f submission.tar.gz -m "{SUBMISSION_MESSAGE}"

## 10. Monitor

In [ ]:
!kaggle competitions submissions orbit-wars